# Processing a whole campaign

A campaign is many raw files, and a large one takes hours. This notebook covers the four
things that make that practical:

1. **One command for many files.** `run_flight.py` takes a list of raw files.
2. **Parallelism that does not change a number.** `--jobs` runs whole files side by side,
   `--s2-jobs` splits one file's decode, and `--chunks` splits one file's framing stage.
   Each is checked byte for byte against the serial run by the test suite. The cell below
   checks it again, on this machine.
3. **Runs that outlive your session**, launched detached.
4. **Resuming** a long framing stage after an interruption.

The input is three short synthetic 2D-S files from the known-truth generator, standing in
for a campaign's files.

The numbering in this folder skips 02: the SODA-2 end-to-end notebook is in the first track, as `1_getting_started/04_soda2_end_to_end`.

**Before this:** [07_reproducible_runs](07_reproducible_runs.ipynb) (Reproducible runs). **Next:** [06_your_own_profile](06_your_own_profile.ipynb). **Needs:** `pip install "arrayprobe[zarr]"` (xarray).

This notebook reads files from the arrayprobe source repository, so run it from a clone of that repository; `pip install arrayprobe` alone does not include them.

In [1]:
# Launch guard: the cells below reach scripts/ and tests/data/ relative to the repository root.
from pathlib import Path

if not (Path("scripts").is_dir() and Path("tests/data").is_dir()):
    raise RuntimeError(
        f"This notebook must run with the repository root as its working directory, not {Path.cwd()}. "
        "Jupyter starts a kernel in the notebook's own folder (examples/2_processing/), so run "
        "`%cd ../..` in a new cell, then run this cell again. "
        "Headless, from the repository root: python scripts/execute_example_notebooks.py --check <name>"
    )

import tempfile

import netCDF4
import numpy as np

from arrayprobe.tools import synthetic_truth as st

tmp = tempfile.TemporaryDirectory()
root = Path(tmp.name)
raw_dir = root / "raw"
raw_dir.mkdir()
raws = []
for k, start in enumerate([(6, 6, 40), (6, 7, 0), (6, 7, 20)]):
    spec = st.FlightSpec(n_seconds=6, rate_per_s=120.0, seed=100 + k, start=start)
    raw = raw_dir / f"base160101{start[0]:02d}{start[1]:02d}{start[2]:02d}.2DS"
    st.build_truth_flight(raw, spec)
    raws.append(raw)
    print(raw.name, f"{raw.stat().st_size / 1e3:.0f} kB")

# One aircraft airspeed record covering all three files (v3.5 times its size distribution on
# it): 06:06:30 to 06:07:40 at 130 m/s.
secs = 6 * 3600 + 6 * 60 + 30 + np.arange(70)
aircraft = root / "aircraft.cdf"
with netCDF4.Dataset(str(aircraft), "w", format="NETCDF3_CLASSIC") as ds:
    ds.createDimension("time", secs.size)
    ds.createVariable("Time", "f8", ("time",))[:] = (
        (secs // 3600) * 10000 + (secs % 3600 // 60) * 100 + secs % 60).astype(float)
    ds.createVariable("TAS", "f8", ("time",))[:] = 130.0

base160101060640.2DS 53 kB
base160101060700.2DS 53 kB
base160101060720.2DS 53 kB


## One command, run serially and in parallel

The same three files, the same engine and preset, twice: once serially, once with every
kind of parallelism on. `--jit` compiles the framing stage's inner loops with numba when
it is installed, and is byte-identical too.

In [2]:
import importlib.util
import subprocess
import sys
import time

common = [sys.executable, "scripts/run_flight.py", *map(str, raws), "--engine", "3.5",
          "--preset", "DEMO_2DS_H", "--profile", "stock-v3.5", "--tas-file", str(aircraft), "--stages", "234"]
parallel = ["--jobs", "3", "--s2-jobs", "2", "--chunks", "2"]
if importlib.util.find_spec("numba") is not None:
    parallel.append("--jit")

timings = {}
for label, extra in (("serial", []), ("parallel", parallel)):
    out = root / label
    t0 = time.perf_counter()
    done = subprocess.run([*common, "--out", str(out), *extra], capture_output=True, text=True)
    timings[label] = time.perf_counter() - t0
    if done.returncode != 0:
        raise RuntimeError((done.stdout + done.stderr)[-3000:])
    print(f"{label:8s} {' '.join(extra) or '(no flags)':40s} exit {done.returncode}, "
          f"{timings[label]:.1f} s")

serial   (no flags)                               exit 0, 13.3 s
parallel --jobs 3 --s2-jobs 2 --chunks 2 --jit    exit 0, 15.9 s


On files this small the parallel run is not faster: starting worker processes costs more
than it saves. On a real flight (hundreds of MB to tens of GB per file) the framing stage
dominates, and `--chunks` and `--jobs` divide it across cores.

## Byte for byte

Every variable of every product, compared between the two runs. The one attribute that
differs by design, `Creation Time`, is left out.

In [3]:
import xarray as xr


def variables(path):
    with xr.open_dataset(path, mask_and_scale=False, decode_times=False) as ds:
        return {name: np.asarray(v.values) for name, v in ds.variables.items()}


checked = 0
for product in sorted((root / "serial").rglob("*.nc")) + sorted((root / "serial").rglob("*.cdf")):
    twin = root / "parallel" / product.relative_to(root / "serial")
    a, b = variables(product), variables(twin)
    assert a.keys() == b.keys(), product.name
    for name in a:
        assert np.array_equal(a[name], b[name], equal_nan=a[name].dtype.kind == "f"), \
            f"{product.name}:{name} differs"
    checked += 1
print(f"{checked} products identical in every variable")

9 products identical in every variable


## Runs that outlive the session

A full campaign runs for hours. Run it in the background with its output written to a log,
not from a notebook cell or the editor, so that closing either does not kill it:

```
python scripts/run_flight.py RAW_DIR/*.2DS --engine 3.5 --preset NAME --profile stock-v3.5 \
    --tas-source auto --tas-search AIRCRAFT_DIR --jobs 8 --chunks 4 --jit --out OUT \
    > my_campaign.log 2>&1 &
```

Check the end of your own log: until it holds the run's final line and the process has
exited, the run has not ended. Write everything to the log and search it afterwards;
piping a background run through another command can stall it.

## Resuming

`--chunks` writes each chunk of the framing stage as its own file before merging them.
If a run is interrupted, `--resume-chunks` keeps the chunks that finished and re-runs only
the rest; the merged result is the same bytes as an uninterrupted run. A worker that stops
making progress for `--worker-stall-min` minutes (default 30) is stopped and reported, so
a stuck file cannot hold a campaign for ever.

## Choosing the numbers

- `--jobs`: up to the machine's physical cores, when there are at least that many files.
- `--chunks`: for a flight with one very large file, where `--jobs` has nothing to spread.
- `--s2-jobs`: the decode stage of one large file.
- Disk, not CPU, is often the limit: every stage writes its own file beside the raw one
  (the cell below lists them). `run_flight.py` estimates the space a file list needs
  before it starts.

In [4]:
raw_kb = sum(r.stat().st_size for r in raws) / 1e3
written_kb = sum(p.stat().st_size for p in (root / "serial").rglob("*") if p.is_file()) / 1e3
print(f"raw input {raw_kb:.0f} kB; written by the serial run {written_kb:.0f} kB")
for p in sorted((root / "serial").rglob("*060640*")):
    print(f"  {p.name:45s} {p.stat().st_size / 1e3:8.1f} kB")

raw input 160 kB; written by the serial run 2932 kB
  base160101060640+2.load.json                       2.7 kB
  base160101060640.H.nc                            397.9 kB
  base160101060640.H.nc.call                         0.2 kB
  base160101060640.H.nc.decode                       0.0 kB
  base160101060640.NV.csv                            2.7 kB
  base160101060640.tas.csv                           0.7 kB
  proc.base160101060640.H.nc                       291.7 kB
  proc.base160101060640.H.nc.call                    0.3 kB
  sd.base160101060640.H.nc                         281.9 kB
  sd.base160101060640.H.nc.call                      0.6 kB
  sd.base160101060640.H.nc.tasqc.json                0.7 kB
